# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Optional stretch — retired 2026-07-13.** Its core already lives in ML-04's data contract (field classification, verified exclusions) and ML-09's leakage attack test. This notebook re-runs both concisely rather than duplicating the full depth of those two.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Same feature set ML-08/09/10 train on, built from `scripts/ml_utils.py`'s Feature bucket.

In [1]:
import sys
sys.path.insert(0, "scripts")
import numpy as np
import pandas as pd
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

numeric_features = list(MODEL_NUMERIC_FEATURES) + ["has_keyword_data", "has_word_count", "has_scroll_data"]
for col in numeric_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_features = list(MODEL_CATEGORICAL_FEATURES)
for col in categorical_features:
    df[col] = df[col].fillna("unknown").astype(str)

X = pd.concat([df[numeric_features], pd.get_dummies(df[categorical_features], prefix=categorical_features)], axis=1)
print("feature vector shape:", X.shape)


feature vector shape: (30000, 55)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Every one of these fields is knowable at export time (a static 90-day-trailing snapshot), so "available before prediction" reduces to "is it label-derived or a product decision, not a timing question" — ML-04's full field-by-field table has the complete reasoning. Summary: 18 numeric + 8 one-hot categorical features = visibility, position, age, content depth, and their missingness flags. Missing values in `search_volume`/`word_count`/`scroll_rate` are flagged (`has_*`) rather than silently zero-filled, since ML-04 showed they go blank by `content_type`, not at random.

In [2]:
print("numeric features:", numeric_features)
print("categorical features:", categorical_features)
print("\nmissingness by content_type (search_volume), the pattern that motivated has_keyword_data:")
print(df.groupby("content_type")["has_keyword_data"].apply(lambda s: round(1 - s.mean(), 3)))


numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'has_keyword_data', 'has_word_count', 'has_scroll_data']
categorical features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

missingness by content_type (search_volume), the pattern that motivated has_keyword_data:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: has_keyword_data, dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Same attack test as ML-09: add the label's excluded source column back in and confirm the score collapses toward 1.0. A test that can't catch a known leak isn't a test.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from ml_utils import precision_at_k

y = df["is_declining_label"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])
honest_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
honest_model.fit(X_train, y.iloc[train_idx])
p_honest = honest_model.predict_proba(X_test)[:, 1]
print("honest precision@50:", round(precision_at_k(y.iloc[test_idx], p_honest, 50), 3))

df["trend_pct_filled"] = df["trend_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)
X_attack = X.copy()
X_attack["trend_pct"] = df["trend_pct_filled"]
attack_numeric = numeric_features + ["trend_pct"]
Xa_train, Xa_test = X_attack.iloc[train_idx].copy(), X_attack.iloc[test_idx].copy()
scaler_a = StandardScaler()
Xa_train[attack_numeric] = scaler_a.fit_transform(Xa_train[attack_numeric])
Xa_test[attack_numeric] = scaler_a.transform(Xa_test[attack_numeric])
attack_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
attack_model.fit(Xa_train, y.iloc[train_idx])
p_attack = attack_model.predict_proba(Xa_test)[:, 1]
print("attack precision@50 (trend_pct added back in):", round(precision_at_k(y.iloc[test_idx], p_attack, 50), 3),
      "-- confirms the harness detects a real leak")


honest precision@50: 0.7


attack precision@50 (trend_pct added back in): 1.0 -- confirms the harness detects a real leak


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `trend_direction`, `trend_pct` — define the label itself.
- `impressions_last_30d`/`prev_30d` and their click/session siblings — proven in ML-07 to be `trend_pct`'s literal formula inputs (100% recompute match), not just correlated.
- `provider_used`, `model_used` — content-generation metadata, not a performance signal.
- Raw `impressions_90d`/`clicks_90d`/`sessions_90d`/`ai_sessions_90d` — replaced by `log1p` versions (heavy-tailed).
- `content_id`, `client_id` — grouping/splitting only, never features.

In [4]:
# Confirms none of the excluded columns above ended up in X by accident
excluded = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
            "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
            "provider_used", "model_used", "content_id", "client_id",
            "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"}
print("excluded columns leaking into the feature vector (should be empty):", sorted(excluded & set(X.columns)))


excluded columns leaking into the feature vector (should be empty): []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.